<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-detection.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>Object Detection (Penn-Fudan)</b> notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> traces training signals back to the exact samples producing them. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Object Detection (Penn-Fudan) with WeightsLab

This notebook trains a small detector (MobileNetV3-Small backbone + a lightweight detection head) on the **Penn-Fudan** pedestrian dataset and instruments it with WeightsLab, so every training signal is traced **back to the exact samples** producing it.

WeightsLab records **per-sample** detection loss and **per-instance** IoU (one value per ground-truth box) live, so each predicted box is traceable in Studio - you can rank the worst samples, spot bad boxes, and curate the dataset **without restarting training**.

> Data: the Penn-Fudan dataset (~170 images, one class: *person*) is **downloaded automatically on first run** - nothing to upload.

### What you'll do
1. Install WeightsLab.
2. Inline the dataset, detector, and loss/IoU criterions (previously the example's `utils/` package).
3. Set every knob in one **config** dict (like a `config.yaml`).
4. Wrap the model, optimizer, dataloaders, loss, and IoU metrics with the SDK.
5. Train while per-sample and per-instance signals are captured, then open the live **Weights Studio** UI.

## Setup

On Colab, pick a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install -q weightslab
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev3"

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase. Every external dependency used by the inlined modules is imported here.

In [ ]:
import os
import ssl
import zipfile
import logging
import tempfile
import urllib.request

import numpy as np
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from PIL import Image

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
logging.getLogger("PIL").setLevel(logging.INFO)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Inlined helper modules (from the example's `utils/`)

The example ships a small `utils/` package; its three modules are inlined below so the notebook is fully self-contained. Only the intra-package imports are dropped - those names (`decode_grid`, `det_collate`, the dataset and the criterions) are now notebook-global.

- **`utils/data.py`** - the Penn-Fudan detection dataset (auto-downloads on first use) + the `det_collate` collate fn.
- **`utils/model.py`** - a MobileNetV3-Small backbone + a small grid detection head.
- **`utils/criterions.py`** - per-sample YOLO-style detection loss and per-sample / per-instance IoU.

In [ ]:
# ===== utils/data.py =====
import os
import ssl
import zipfile
import urllib.request

import numpy as np
import torch

from torchvision import transforms

from torch.utils.data import Dataset

from PIL import Image


# =============================================================================
# Penn-Fudan Pedestrian detection dataset
# =============================================================================
# A small, real object-detection dataset (~170 photos, one class: "person").
# It ships per-instance segmentation masks; we derive an axis-aligned bounding
# box per pedestrian from each mask. Downloaded + extracted on first use.
#
# On-disk layout after extraction:
# <root>/PennFudanPed/
# PNGImages/FudanPed00001.png ...
# PedMasks/FudanPed00001_mask.png ... # pixel value k = k-th pedestrian, 0 = bg
#
# WL renders detection targets/predictions from a per-sample [N, 6] array
# ``[x1, y1, x2, y2, class_id, confidence]`` normalized to [0, 1] (GT conf = 1.0)
# — see ``get_items`` below and ``data_service.py`` (task_type == "detection").

CLASS_NAMES = ["person"]

_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

# ImageNet statistics — the MobileNetV3 backbone is pretrained with these, so we
# normalize model inputs the same way. (The UI still shows the original PNG.)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def download_penn_fudan(root):
    """Download + extract Penn-Fudan into <root>/PennFudanPed (idempotent)."""
    base = os.path.join(root, "PennFudanPed")
    if os.path.isdir(os.path.join(base, "PNGImages")):
        return base

    os.makedirs(root, exist_ok=True)
    zip_path = os.path.join(root, "PennFudanPed.zip")

    if not os.path.exists(zip_path):
        print(f"[data] Downloading Penn-Fudan dataset to {zip_path} ...", flush=True)
        try:
            urllib.request.urlretrieve(_URL, zip_path)
        except Exception as e:
            # Some corporate environments break TLS verification - retry unverified.
            print(f"[data] TLS verification failed ({e}); retrying without verification.", flush=True)
            ctx = ssl._create_unverified_context()
            with urllib.request.urlopen(_URL, context=ctx) as resp, open(zip_path, "wb") as fh:
                fh.write(resp.read())

    print("[data] Extracting Penn-Fudan ...", flush=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(root)
    return base


def _boxes_from_mask(mask_path):
    """Derive one bbox per pedestrian from a Penn-Fudan instance mask.

    Returns (boxes_px [N, 4] int xyxy, height, width). Background (0) skipped.
    """
    mask = np.array(Image.open(mask_path))
    h, w = mask.shape[:2]
    obj_ids = np.unique(mask)
    obj_ids = obj_ids[obj_ids != 0] # drop background

    boxes = []
    for oid in obj_ids:
        ys, xs = np.where(mask == oid)
        if xs.size == 0:
            continue
        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return np.asarray(boxes, dtype=np.float32).reshape(-1, 4), h, w


class PennFudanDetectionDataset(Dataset):
    """Pedestrian bounding-box detection over the Penn-Fudan images.

    Args:
        root: directory to download/extract the dataset into.
        split: "train" or "val" (deterministic split of the 170 images).
        image_size: square resize fed to the model.
        val_fraction: fraction of images held out for validation.
        max_samples: optional cap on the split size (for quick runs).
    """

    def __init__(
        self,
        root,
        split="train",
        num_classes=1,
        image_size=256,
        val_fraction=0.2,
        max_samples=None,
    ):
        super().__init__()
        self.root = root
        self.split = split
        self.num_classes = num_classes
        self.image_size = image_size
        # Explicit task type; bypasses WL's label-shape heuristic so bboxes are
        # rendered as detection overlays (not mistaken for classification).
        self.task_type = "detection"
        self.class_names = CLASS_NAMES[:num_classes]

        base = download_penn_fudan(root)
        img_dir = os.path.join(base, "PNGImages")
        mask_dir = os.path.join(base, "PedMasks")

        all_imgs = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(".png"))

        # Deterministic train/val split: every k-th image goes to val.
        k = max(2, int(round(1.0 / max(val_fraction, 1e-6))))
        if split == "val":
            selected = all_imgs[::k]
        else:
            val_set = set(all_imgs[::k])
            selected = [f for f in all_imgs if f not in val_set]

        selected = selected[:max_samples] if max_samples != None else selected

        self.images = []
        self.masks = []
        for fname in selected:
            base_name, _ = os.path.splitext(fname)
            mask_path = os.path.join(mask_dir, base_name + "_mask.png")
            if os.path.exists(mask_path):
                self.images.append(os.path.join(img_dir, fname))
                self.masks.append(mask_path)

        if len(self.images) == 0:
            raise RuntimeError(f"No image/mask pairs found under {base}")

        self.image_transform = transforms.Compose(
            [
                transforms.Resize((image_size, image_size), interpolation=Image.BILINEAR),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        )

    def __len__(self):
        return len(self.images)

    def _load_boxes(self, mask_path):
        """Read mask → [N, 6] float32 = [x1, y1, x2, y2, cls=0, conf=1.0] (normalized)."""
        boxes_px, h, w = _boxes_from_mask(mask_path)
        if boxes_px.shape[0] == 0:
            return np.zeros((0, 6), dtype=np.float32)
        norm = boxes_px.copy()
        norm[:, [0, 2]] /= float(w)
        norm[:, [1, 3]] /= float(h)
        n = norm.shape[0]
        cls = np.zeros((n, 1), dtype=np.float32) # single class: person
        conf = np.ones((n, 1), dtype=np.float32)
        return np.concatenate([norm, cls, conf], axis=1).astype(np.float32)

    def __getitem__(self, idx):
        """Returns (item, uid, target, metadata).

        - item: normalized image tensor [C, H, W]
        - uid: unique sample id (string)
        - target: [N, 6] float32 = [x1, y1, x2, y2, class_id, confidence]
        - metadata: dict with source paths
        """
        return self.get_items(idx, include_metadata=True, include_labels=True, include_images=True)

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):
        img_path = self.images[idx]
        mask_path = self.masks[idx]
        uid = os.path.splitext(os.path.basename(img_path))[0]

        metadata = {
            "img_path": img_path,
            "mask_path": mask_path,
        } if include_metadata else None

        img_t = None
        if include_images:
            img = Image.open(img_path).convert("RGB")
            img_t = self.image_transform(img)

        target = None
        if include_labels:
            target = self._load_boxes(mask_path)

        return img_t, uid, target, metadata


def det_collate(batch):
    """Collate WL per-sample tuples for object detection.

    A sample owns a variable number of boxes, so targets cannot be stacked. We
    keep them as a Python list (one [N_i, 6] tensor per sample), exactly the
    layout WL's per-instance helpers expect (``targets[s]`` iterates that
    sample's boxes in annotation order).

    Returns:
        images: FloatTensor [B, C, H, W]
        ids: list[str] of length B
        targets: list[B] of [N_i, 6] float tensors ([x1, y1, x2, y2, cls, conf])
        metas: list[B] of metadata dicts
    """
    images = torch.stack([b[0] for b in batch], dim=0)
    ids = [b[1] for b in batch]
    targets = [
        torch.as_tensor(b[2], dtype=torch.float32)
        if not isinstance(b[2], torch.Tensor) else b[2].float()
        for b in batch
    ]
    metas = [b[3] if len(b) > 3 else None for b in batch]
    return images, ids, targets, metas

In [ ]:
# ===== utils/model.py =====
# =============================================================================
# Small single-shot grid detector on a pretrained MobileNetV3-Small backbone
# =============================================================================
# A frozen/fine-tuned ImageNet-pretrained backbone extracts features; a tiny
# detection head turns them into an S x S grid where each cell predicts ONE box:
# (objectness, tx, ty, tw, th, class_logits...).
#
# Encoding (all coordinates normalized to the [0, 1] image frame):
# * objectness = sigmoid(t_obj) -> P(box present in this cell)
# * cx = (col + sigmoid(tx)) / S -> box center, x
# * cy = (row + sigmoid(ty)) / S -> box center, y
# * w = sigmoid(tw) -> box width (fraction of image)
# * h = sigmoid(th) -> box height (fraction of image)
# * class = softmax(class_logits)
#
# Raw forward output keeps logits (loss applies the activations); `decode`
# turns logits into xyxy boxes for metrics and UI rendering.
import torch
import torch.nn as nn

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


def decode_grid(outputs, grid_size):
    """Decode raw grid logits -> per-cell xyxy boxes, objectness and class probs.

    Shared by the model (``SmallDetector.decode``) and the criterions so the
    encoding lives in exactly one place.

    Args:
        outputs: [B, S, S, 5 + num_classes] raw logits.
        grid_size: S.

    Returns:
        boxes: [B, S, S, 4] xyxy in [0, 1]
        obj: [B, S, S] objectness probability
        cls_probs: [B, S, S, num_classes] class probabilities
    """
    B, S, _, _ = outputs.shape
    device = outputs.device

    obj = torch.sigmoid(outputs[..., 0])
    tx = torch.sigmoid(outputs[..., 1])
    ty = torch.sigmoid(outputs[..., 2])
    w = torch.sigmoid(outputs[..., 3])
    h = torch.sigmoid(outputs[..., 4])
    cls_probs = torch.softmax(outputs[..., 5:], dim=-1)

    # Cell-origin grid (col=x, row=y).
    cols = torch.arange(S, device=device).view(1, 1, S).expand(B, S, S)
    rows = torch.arange(S, device=device).view(1, S, 1).expand(B, S, S)

    cx = (cols + tx) / S
    cy = (rows + ty) / S
    x1 = (cx - w / 2).clamp(0, 1)
    y1 = (cy - h / 2).clamp(0, 1)
    x2 = (cx + w / 2).clamp(0, 1)
    y2 = (cy + h / 2).clamp(0, 1)

    boxes = torch.stack([x1, y1, x2, y2], dim=-1)
    return boxes, obj, cls_probs


class SmallDetector(nn.Module):
    def __init__(
        self,
        in_channels=3,
        num_classes=1,
        image_size=256,
        grid_size=8,
        pretrained=True,
        freeze_backbone=True,
    ):
        super().__init__()
        # For WeightsLab
        self.task_type = "detection"
        self.num_classes = num_classes
        self.class_names = ["person"][:num_classes]
        self.grid_size = grid_size
        self.image_size = image_size
        self.input_shape = (1, in_channels, image_size, image_size)

        # Channels per cell: objectness(1) + box(4) + class logits(num_classes)
        self.preds_per_cell = 5 + num_classes

        # --- Pretrained backbone (ImageNet) ---
        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = mobilenet_v3_small(weights=weights)
        self.backbone = backbone.features # [B, 576, H/32, W/32]
        backbone_out_ch = 576

        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # --- Detection head (lightweight, trained from scratch) ---
        self.neck = nn.Sequential(
            nn.Conv2d(backbone_out_ch, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.head = nn.Conv2d(128, self.preds_per_cell, 1)

    def train(self, mode=True):
        """Keep a frozen backbone in eval mode (so its BatchNorm stats stay fixed)."""
        super().train(mode)
        if self.freeze_backbone:
            self.backbone.eval()
        return self

    def forward(self, x):
        """Returns raw logits [B, S, S, 5 + num_classes]."""
        if self.freeze_backbone:
            with torch.no_grad():
                feat = self.backbone(x)
        else:
            feat = self.backbone(x)

        feat = self.neck(feat)
        out = self.head(feat) # [B, preds_per_cell, S', S']

        # Resize feature grid to the configured grid_size.
        if out.shape[-1] != self.grid_size or out.shape[-2] != self.grid_size:
            out = nn.functional.adaptive_avg_pool2d(out, self.grid_size)
        # [B, C, S, S] -> [B, S, S, C]
        return out.permute(0, 2, 3, 1).contiguous()

    def decode(self, outputs):
        """Decode raw logits -> per-cell xyxy boxes, objectness, class probs."""
        return decode_grid(outputs, self.grid_size)

In [ ]:
# ===== utils/criterions.py =====
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# Per-instance / per-sample detection criterions (YOLO-v1 style loss + IoU)
# =============================================================================
# The detection dataset yields, per sample, a [N, 6] target tensor
# ``[x1, y1, x2, y2, class_id, confidence]`` (normalized to [0, 1]). Each GT box
# is assigned to the grid cell containing its center; that cell is "responsible"
# for predicting the box.
#
# * PerSampleDetectionLoss -> one differentiable loss scalar per sample ([B]),
# wrapped with ``per_sample=True`` (the value WL backprops + dashboards).
# * PerSampleIoU -> mean IoU over a sample's boxes ([B]), a metric.
# * PerInstanceIoU -> flat tensor of one IoU per GT box (sample-major
# order), wrapped with ``per_instance=True`` so WL auto-saves it at
# (sample_id, annotation_id). The ordering matches the per-sample target
# iteration, so the wrapper's auto ``batch_idx`` maps each value correctly.

_EPS = 1e-6
_LAMBDA_COORD = 5.0
_LAMBDA_NOOBJ = 0.5


def box_iou_xyxy(a, b):
    """IoU between two aligned sets of xyxy boxes. a, b: [..., 4] -> [...]."""
    x1 = torch.maximum(a[..., 0], b[..., 0])
    y1 = torch.maximum(a[..., 1], b[..., 1])
    x2 = torch.minimum(a[..., 2], b[..., 2])
    y2 = torch.minimum(a[..., 3], b[..., 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_a = (a[..., 2] - a[..., 0]).clamp(min=0) * (a[..., 3] - a[..., 1]).clamp(min=0)
    area_b = (b[..., 2] - b[..., 0]).clamp(min=0) * (b[..., 3] - b[..., 1]).clamp(min=0)
    union = area_a + area_b - inter + _EPS
    return inter / union


def _responsible_cells(boxes, S):
    """Map GT boxes -> their responsible (row, col) cell and center offsets.

    Args:
        boxes: [N, 4] xyxy in [0, 1].
        S: grid size.

    Returns:
        rows, cols: [N] long, the responsible cell indices.
        off_x, off_y: [N] center offset within the cell, in [0, 1).
        w, h: [N] box size as a fraction of the image.
    """
    cx = (boxes[:, 0] + boxes[:, 2]) / 2
    cy = (boxes[:, 1] + boxes[:, 3]) / 2
    w = (boxes[:, 2] - boxes[:, 0]).clamp(_EPS, 1.0)
    h = (boxes[:, 3] - boxes[:, 1]).clamp(_EPS, 1.0)

    cols = (cx * S).long().clamp(0, S - 1)
    rows = (cy * S).long().clamp(0, S - 1)
    off_x = (cx * S - cols).clamp(0, 1)
    off_y = (cy * S - rows).clamp(0, 1)
    return rows, cols, off_x, off_y, w, h


def _per_sample_loss(outputs, targets, num_classes, weights=None):
    """YOLO-v1 style loss, returned as one scalar per sample ([B], with grad)."""
    B, S = outputs.shape[0], outputs.shape[1]
    device = outputs.device

    obj_logit = outputs[..., 0] # [B, S, S]
    tx = torch.sigmoid(outputs[..., 1])
    ty = torch.sigmoid(outputs[..., 2])
    w_pred = torch.sigmoid(outputs[..., 3])
    h_pred = torch.sigmoid(outputs[..., 4])
    cls_logits = outputs[..., 5:] # [B, S, S, C]

    if weights is not None:
        weights = torch.as_tensor(weights, device=device, dtype=outputs.dtype)

    losses = []
    for s in range(B):
        tgt = targets[s]
        tgt = torch.as_tensor(tgt, device=device, dtype=outputs.dtype)
        if tgt.ndim == 1:
            tgt = tgt.view(-1, 6) if tgt.numel() else tgt.view(0, 6)

        obj_target = torch.zeros((S, S), device=device, dtype=outputs.dtype)

        coord_loss = torch.zeros((), device=device)
        class_loss = torch.zeros((), device=device)

        if tgt.numel() > 0:
            boxes = tgt[:, :4]
            cls_ids = tgt[:, 4].long().clamp(0, num_classes - 1)
            rows, cols, off_x, off_y, gw, gh = _responsible_cells(boxes, S)

            obj_target[rows, cols] = 1.0

            # Localization: center offset (linear) + size in sqrt space (YOLO trick
            # so small-box errors weigh as much as large-box ones).
            coord = (
                (tx[s, rows, cols] - off_x) ** 2
                + (ty[s, rows, cols] - off_y) ** 2
                + (torch.sqrt(w_pred[s, rows, cols] + _EPS) - torch.sqrt(gw + _EPS)) ** 2
                + (torch.sqrt(h_pred[s, rows, cols] + _EPS) - torch.sqrt(gh + _EPS)) ** 2
            )
            coord_loss = _LAMBDA_COORD * coord.sum()

            ce = F.cross_entropy(
                cls_logits[s, rows, cols], cls_ids, reduction="none"
            )
            if weights is not None:
                ce = ce * weights[cls_ids]
            class_loss = ce.sum()

        # Objectness BCE over the whole grid; down-weight the (many) empty cells.
        obj_weight = torch.where(
            obj_target > 0,
            torch.ones_like(obj_target),
            torch.full_like(obj_target, _LAMBDA_NOOBJ),
        )
        obj_loss = (
            F.binary_cross_entropy_with_logits(
                obj_logit[s], obj_target, reduction="none"
            ) * obj_weight
        ).sum()

        losses.append(coord_loss + class_loss + obj_loss)

    return torch.stack(losses)


def _per_box_iou(outputs, targets, grid_size):
    """IoU of each GT box against its responsible cell's decoded prediction.

    Returns a list[B] of 1-D tensors (one IoU per box for that sample, in
    annotation order). Detached — this is a metric, not a loss.
    """
    boxes_grid, _, _ = decode_grid(outputs, grid_size) # [B, S, S, 4]
    B = outputs.shape[0]
    S = grid_size
    device = outputs.device

    per_sample = []
    for s in range(B):
        tgt = torch.as_tensor(targets[s], device=device, dtype=outputs.dtype)
        if tgt.numel() == 0:
            per_sample.append(torch.zeros(0, device=device))
            continue
        if tgt.ndim == 1:
            tgt = tgt.view(-1, 6)

        gt_boxes = tgt[:, :4]
        rows, cols, _, _, _, _ = _responsible_cells(gt_boxes, S)
        pred_boxes = boxes_grid[s, rows, cols] # [N, 4]
        ious = box_iou_xyxy(pred_boxes, gt_boxes) # [N]
        per_sample.append(ious.detach())

    return per_sample


class PerSampleDetectionLoss(nn.Module):
    """Total detection loss aggregated to one value per sample ([B])."""

    def __init__(self, num_classes, grid_size, weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.register_buffer(
            "weights", torch.tensor(weights) if weights is not None else None
        )

    def forward(self, outputs, targets):
        return _per_sample_loss(outputs, targets, self.num_classes, weights=self.weights)


class PerSampleIoU(nn.Module):
    """Mean IoU over a sample's boxes -> one value per sample ([B])."""

    def __init__(self, num_classes, grid_size):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size

    def forward(self, outputs, targets):
        per_sample = _per_box_iou(outputs, targets, self.grid_size)
        out = [
            v.mean() if v.numel() > 0 else torch.zeros((), device=outputs.device)
            for v in per_sample
        ]
        return torch.stack(out).detach()


class PerInstanceIoU(nn.Module):
    """IoU per GT box -> flat tensor [total_boxes] (sample-major order)."""

    def __init__(self, num_classes, grid_size):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size

    def forward(self, outputs, targets):
        per_sample = _per_box_iou(outputs, targets, self.grid_size)
        flat = [v for s in per_sample for v in s]
        if not flat:
            return torch.zeros(0, device=outputs.device)
        return torch.stack(flat).detach()


# =============================================================================
# Inference-time decoding (for UI prediction overlays)
# =============================================================================
def decode_predictions(outputs, grid_size, conf_thresh=0.3, max_det=10):
    """Turn raw grid logits into a per-sample list of detected boxes.

    Returns list[B] of [M, 6] numpy-friendly tensors
    ``[x1, y1, x2, y2, class_id, confidence]`` (kept on CPU, detached) — the
    exact 6-column schema WL renders for detection predictions.
    """
    boxes_grid, obj, cls_probs = decode_grid(outputs, grid_size)
    B, S = outputs.shape[0], grid_size

    cls_conf, cls_id = cls_probs.max(dim=-1) # [B, S, S]
    score = obj * cls_conf # combined confidence

    flat_boxes = boxes_grid.view(B, S * S, 4)
    flat_score = score.view(B, S * S)
    flat_cls = cls_id.view(B, S * S)

    results = []
    for s in range(B):
        keep = flat_score[s] >= conf_thresh
        if keep.sum() == 0:
            results.append(torch.zeros((0, 6)))
            continue
        sc = flat_score[s][keep]
        bx = flat_boxes[s][keep]
        cl = flat_cls[s][keep].to(bx.dtype)

        # Keep the most confident detections (cheap top-k in place of full NMS).
        order = torch.argsort(sc, descending=True)[:max_det]
        det = torch.cat([bx[order], cl[order, None], sc[order, None]], dim=1)
        results.append(det.detach().cpu())

    return results

## 3. Configuration

Every tunable lives here in one dict (the example's `config.yaml`, inlined with its comments). Wrapping it with `flag="hyperparameters"` lets the Studio UI read - and live-edit - these values while training.

In [ ]:
# Inlined from the example's config.yaml (comments preserved).
config = {
    # -- Global -----------------------------------------------------------
    "device": "auto",                          # auto -> cuda if available else cpu (resolved in the imports cell)
    "experiment_name": "detection_pennfudan_usecase",  # name shown in Weights Studio
    "training_steps_to_do": None,              # None = open-ended in the example; capped below for Colab
    "root_log_dir": None,                      # None -> a temp dir is created below

    "checkpoint_manager": {
        "load_config": False,                  # ignore a previous run's config so edits here take effect
    },

    # Initially compute natural-sort signals, e.g. average intensity.
    "compute_natural_sort": True,

    # -- Experiment schedule ---------------------------------------------
    "eval_full_to_train_steps_ratio": 5,       # run a full eval every N steps
    "experiment_dump_to_train_steps_ratio": 5, # export history + dataframe every N steps
    "tqdm_display": True,
    "is_training": False,                      # start training immediately or not

    # -- Serving ----------------------------------------------------------
    "serving_grpc": True,                      # expose the gRPC backend for Weights Studio
    "serving_cli": True,

    # -- Global dataframe storage ----------------------------------------
    "ledger_enable_h5_persistence": True,
    "ledger_enable_flushing_threads": True,
    "ledger_flush_max_rows": 100,
    "ledger_flush_interval": 60.0,

    # -- Model / task -----------------------------------------------------
    "num_classes": 1,                          # Penn-Fudan: single class (person)
    "image_size": 256,
    "grid_size": 8,                            # detector predicts on an 8x8 cell grid
    "conf_thresh": 0.3,                        # objectness*class confidence threshold for displayed predictions
    "pretrained_backbone": True,               # ImageNet-pretrained MobileNetV3-Small feature extractor
    "freeze_backbone": True,                   # train only the detection head (faster, less data hungry)

    "optimizer": {
        "lr": 0.001,
    },

    # -- Data (Penn-Fudan pedestrians; ~170 images downloaded on first run under data_root) --
    "data_root": "./data",                     # portable relative path; the dataset downloads here on first run
    "data": {
        "train_loader": {
            "batch_size": 8,
            "shuffle": True,
            "max_samples": None,               # None = use the full training split
        },
        "test_loader": {
            "batch_size": 8,
            "shuffle": False,
            "max_samples": None,
        },
    },
}

wl.watch_or_edit(config, flag="hyperparameters", defaults=config, poll_interval=1.0)

# Resolve the log directory (a temp dir when unset), mirroring the example's main.py.
log_dir = config.get("root_log_dir") or tempfile.mkdtemp(prefix="weightslab_detection_")
config["root_log_dir"] = log_dir
os.makedirs(log_dir, exist_ok=True)
print("Experiment logs ->", log_dir)

## 4. Data

Build the Penn-Fudan train/val datasets (the ~170 images **download automatically** on first construction) and wrap them as tracked dataloaders. Detection targets have a variable number of boxes per image, so the custom `det_collate` keeps them as a list.

In [ ]:
# Penn-Fudan auto-downloads (~170 images) on first dataset construction - nothing to upload.
data_root = config.get("data_root") or "./data"
os.makedirs(data_root, exist_ok=True)

num_classes = int(config["num_classes"])
image_size = int(config["image_size"])
grid_size = int(config["grid_size"])
conf_thresh = float(config["conf_thresh"])

train_cfg = config.get("data", {}).get("train_loader", {})
test_cfg = config.get("data", {}).get("test_loader", {})

_train_dataset = PennFudanDetectionDataset(
    root=data_root,
    split="train",
    num_classes=num_classes,
    image_size=image_size,
    max_samples=train_cfg.get("max_samples", None),
)
_val_dataset = PennFudanDetectionDataset(
    root=data_root,
    split="val",
    num_classes=num_classes,
    image_size=image_size,
    max_samples=test_cfg.get("max_samples", None),
)

# Tracked dataloaders (detection uses the custom det_collate defined above).
train_loader = wl.watch_or_edit(
    _train_dataset,
    flag="data",
    loader_name="train_loader",
    batch_size=train_cfg.get("batch_size", 8),
    shuffle=train_cfg.get("shuffle", True),
    compute_hash=False,
    is_training=True,
    array_autoload_arrays=False,
    array_return_proxies=True,
    array_use_cache=True,
    preload_labels=False,
    collate_fn=det_collate,
)
test_loader = wl.watch_or_edit(
    _val_dataset,
    flag="data",
    loader_name="test_loader",
    batch_size=test_cfg.get("batch_size", 8),
    shuffle=test_cfg.get("shuffle", False),
    compute_hash=False,
    is_training=False,
    array_autoload_arrays=False,
    array_return_proxies=True,
    array_use_cache=True,
    preload_labels=True,
    collate_fn=det_collate,
)

## 5. Model, optimizer and tracked signals

Wrap the detector, optimizer, loss and IoU metrics with `wl.watch_or_edit(...)`. The loss is tracked **per sample**; per-instance IoU stores one value per ground-truth box at `(sample_id, annotation_id)`.

In [ ]:
# --- Model ---
_model = SmallDetector(
    in_channels=3, num_classes=num_classes,
    image_size=image_size, grid_size=grid_size,
    pretrained=bool(config["pretrained_backbone"]),
    freeze_backbone=bool(config["freeze_backbone"]),
).to(device)
model = wl.watch_or_edit(_model, flag="model", device=device)

# --- Optimizer (only trainable params; the backbone may be frozen) ---
lr = config.get("optimizer", {}).get("lr", 1e-3)
trainable = [p for p in model.parameters() if p.requires_grad]
_optimizer = optim.Adam(trainable, lr=lr)
optimizer = wl.watch_or_edit(_optimizer, flag="optimizer")


# --- Detection loss (per sample) + IoU (per sample & per instance) signals ---
# per_sample=True auto-saves one value per sample; per_instance=True auto-saves
# one IoU per (sample_id, annotation_id), i.e. one per ground-truth box.
def _make_det_signals(split: str, weights=None) -> dict:
    return {
        "loss": wl.watch_or_edit(
            PerSampleDetectionLoss(num_classes, grid_size, weights=weights),
            flag="loss",
            name=f"{split}_loss/sample", per_sample=True, log=True,
        ),
        "iou_sample": wl.watch_or_edit(
            PerSampleIoU(num_classes, grid_size), flag="metric",
            name=f"{split}_iou/sample", per_sample=True, log=True,
        ),
        "iou_instance": wl.watch_or_edit(
            PerInstanceIoU(num_classes, grid_size), flag="metric",
            name=f"{split}_iou/instance", per_instance=True, log=True,
        ),
    }


# Class weights from ground-truth box counts (optional; balances rare shapes).
def compute_class_weights(dataset, num_classes, max_samples=200):
    print("\n" + "=" * 60, flush=True)
    print(f"Computing class weights for {num_classes} classes (max {max_samples} samples)...", flush=True)
    class_counts = np.zeros(num_classes, dtype=np.float64)
    num_samples = min(len(dataset), max_samples)

    for idx in tqdm.tqdm(range(num_samples), desc=" Analyzing Distribution"):
        _, _, target, _ = dataset.get_items(idx, include_labels=True)
        if target is None or len(target) == 0:
            continue
        for c in target[:, 4].astype(np.int64):
            if 0 <= c < num_classes:
                class_counts[c] += 1

    class_counts = np.maximum(class_counts, 1)  # Avoid div by zero
    total = class_counts.sum()
    class_weights = total / (num_classes * class_counts)
    class_weights = class_weights / class_weights.mean()  # Normalize

    print("\nClass distribution and weights:", flush=True)
    for c in range(num_classes):
        pct = (class_counts[c] / total) * 100
        print(f"Class {c} ({dataset.class_names[c]}): {pct:6.2f}% -> weight: {class_weights[c]:.3f}", flush=True)
    print("=" * 60 + "\n", flush=True)
    return torch.FloatTensor(class_weights).to(device)


weights = compute_class_weights(_train_dataset, num_classes)

train_sig = _make_det_signals("train", weights=weights)
test_sig = _make_det_signals("test", weights=weights)

## 6. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. The watched loss rides along with `batch_ids=ids` (so it is stored per sample) and `preds=` (so the decoded boxes are saved for the UI overlay).

In [ ]:
def train(loader, model, optimizer, sig, device, grid_size, conf_thresh):
    """Single training step using the tracked dataloader + watched loss.

    loader yields (inputs, ids, targets, metadata) because of the
    DataSampleTrackingWrapper. `targets` is per sample a [N, 6] tensor of boxes
    ([x1, y1, x2, y2, class_id, confidence]); see utils/data.det_collate.
    """
    with guard_training_context:
        (inputs, ids, targets, _) = next(loader)
        inputs = inputs.to(device)
        targets = [t.to(device) for t in targets]

        optimizer.zero_grad()
        outputs = model(inputs) # [B, S, S, 5 + num_classes]

        # Decoded boxes for the UI overlay (detached - display only).
        preds = decode_predictions(outputs.detach(), grid_size, conf_thresh=conf_thresh)

        # Per-sample detection loss (tracked, saved, and the value we backprop).
        # `preds=` rides along so WL stores the predicted boxes for this batch.
        loss_per_sample = sig["loss"](outputs, targets, batch_ids=ids, preds=preds)

        # Metrics: per-sample mean IoU + per-instance IoU (one value per box).
        sig["iou_sample"](outputs, targets, batch_ids=ids)
        sig["iou_instance"](outputs, targets, batch_ids=ids)

        loss = loss_per_sample.mean()
        loss.backward()
        optimizer.step()

    return float(loss.detach().cpu().item())


def test(loader, model, sig, device, grid_size, conf_thresh, test_loader_len):
    """Full evaluation pass over the val loader."""
    losses = 0.0
    ious = 0.0
    with guard_testing_context, torch.no_grad():
        for inputs, ids, targets, _ in loader:
            inputs = inputs.to(device)
            targets = [t.to(device) for t in targets]

            outputs = model(inputs)
            preds = decode_predictions(outputs, grid_size, conf_thresh=conf_thresh)

            loss_per_sample = sig["loss"](outputs, targets, batch_ids=ids, preds=preds)
            iou_per_sample = sig["iou_sample"](outputs, targets, batch_ids=ids)
            sig["iou_instance"](outputs, targets, batch_ids=ids)

            losses += torch.mean(loss_per_sample)
            ious += torch.mean(iou_per_sample)

    loss = float((losses / test_loader_len).detach().cpu().item())
    iou = float((ious / test_loader_len).detach().cpu().item())
    return loss, iou * 100.0 # Return mean IoU as percentage

## 7. Serve and train

`wl.serve(...)` starts the background gRPC server (and a public `bore` tunnel) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

Leave this cell **running** while you watch it stream in the UI.

In [ ]:
# Start the background gRPC server + a public bore tunnel so a local Weights Studio can attach.
wl.serve(serving_grpc=True, serving_bore=True)

# Flip the experiment into the training state, then run the loop.
wl.start_training(timeout=3)

# training_steps_to_do is None (open-ended) in the example; cap the demo to a finite
# number of steps so the cell terminates on its own on Colab (raise for a longer run).
max_steps = config["training_steps_to_do"] or 1500
eval_ratio = config["eval_full_to_train_steps_ratio"]
export_ratio = config["experiment_dump_to_train_steps_ratio"]

test_loss, test_metric = None, None
pbar = tqdm.tqdm(range(max_steps), desc="Training")
for train_step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else train_step

    # Train one step.
    train_loss = train(train_loader, model, optimizer, train_sig, device, grid_size, conf_thresh)

    # Full eval pass every eval_ratio steps.
    if age == 0 or age % eval_ratio == 0:
        test_loader_len = len(test_loader)
        test_loss, test_metric = test(test_loader, model, test_sig, device, grid_size, conf_thresh, test_loader_len)

    # Export per-sample history + dataframe periodically.
    if age > 0 and age % export_ratio == 0:
        wl.write_history()
        wl.write_dataframe()

    pbar.set_postfix(
        train_loss=f"{train_loss:.4f}",
        test_loss=f"{test_loss:.4f}" if test_loss is not None else "N/A",
        iou=f"{test_metric:.2f}%" if test_metric is not None else "N/A",
    )

wl.write_history()
wl.write_dataframe()
print("Training complete. Logs at:", log_dir)

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this notebook's
backend using the `bore.pub:<port>` printed in the Expose section:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 - opens http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 - the host:port from the Expose section
```

Then open **http://localhost:5173**. Prefer local-only? Run this example directly on your machine
(`weightslab start example` selects a bundled example) and launch the UI next to it - no tunnel.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>